# 02 · Chunk analysis

Chunk counts, sizes (characters and the chunker's token estimate), overlap, document-to-chunk relationships, embedding coverage and configuration. Only numeric per-chunk metrics are used (no text).

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
ch = pd.read_csv(A.SNAP / "chunk_metrics.csv")
stats = json.loads((A.SNAP / "dataset_statistics.json").read_text())
cfg = EXPERIMENTS["shared"]
active = ch[ch["active"]].copy()
print(f"{len(ch)} chunk rows: {len(active)} active, {int((~ch['active']).sum())} historical")
pd.DataFrame([("embedding model", cfg["embedding_model"] + " · " + cfg["embedding_variant"]),
              ("embedding dimensions", cfg["embedding_dimensions"]),
              ("dimensions present in the database", stats["chunks"]["embedding_dimensions"]),
              ("chunker", json.dumps(cfg["chunker"])), ("vector index", cfg["vector_store"]["index"]),
              ("pgvector", stats["pgvector_version"])], columns=["setting", "value"])

In [ ]:
cov = pd.DataFrame([("active chunks", len(active)), ("active chunks with embedding", int(active["has_embedding"].sum())),
                    ("historical chunks", int((~ch["active"]).sum())),
                    ("historical chunks with embedding", int(ch.loc[~ch["active"], "has_embedding"].sum())),
                    ("chunk strategies", ", ".join(sorted(ch["chunk_strategy"].unique())))], columns=["measure", "value"])
cov

## Chunk size

`chars` is exact. `token_count` is the chunker's heuristic estimate (words × 1.3 + CJK characters), not a tokenizer count - it is labelled as an estimate everywhere.

In [ ]:
desc = active[["chars", "token_count"]].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T
desc.index = ["characters (exact)", "tokens (chunker estimate)"]
desc.round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(active["chars"], bins=np.arange(0, active["chars"].max() + 100, 100), color=A.SINGLE,
             edgecolor=A.SURFACE, linewidth=1.5, zorder=2)
axes[0].axvline(active["chars"].median(), color=A.INK2, linestyle="--", linewidth=1)
axes[0].text(active["chars"].median(), axes[0].get_ylim()[1] * 0.92, f" median {active['chars'].median():.0f}",
             color=A.INK2, fontsize=9)
axes[0].set_title(f"Active chunk size (n = {len(active)})"); axes[0].set_xlabel("characters per chunk"); axes[0].set_ylabel("chunks")
by_lang = [active.loc[active["language"] == l, "chars"] for l in ["en", "ko"]]
bp = axes[1].boxplot(by_lang, tick_labels=[f"English (n={len(by_lang[0])})", f"Korean (n={len(by_lang[1])})"],
                     patch_artist=True, widths=0.45, medianprops={"color": A.INK})
for patch in bp["boxes"]:
    patch.set_facecolor(A.SINGLE); patch.set_alpha(0.55); patch.set_edgecolor(A.INK2)
axes[1].set_title("Chunk size by document language"); axes[1].set_ylabel("characters per chunk")
fig.tight_layout(); print(A.save(fig, "02_chunk_size_distribution"))

## Document-to-chunk relationship

In [ ]:
per_doc = pd.DataFrame(stats["per_document"])
per_doc["label"] = per_doc["title"].map(lambda t: t if len(t) <= 38 else t[:37].rstrip() + "…")
display(per_doc[["title", "language", "document_type", "chunks", "embedded", "pages_metadata", "text_chars"]])
fig, ax = plt.subplots(figsize=(9, 6.2))
p = per_doc.sort_values("chunks")
ax.barh(p["label"], p["chunks"], color=A.SINGLE, height=0.65, zorder=2)
for y, v in enumerate(p["chunks"]):
    ax.text(v + 1, y, str(v), va="center", fontsize=8, color=A.INK)
ax.set_title(f"Active chunks per document (n = {len(per_doc)} documents, {int(per_doc['chunks'].sum())} chunks)")
ax.set_xlabel("chunks"); ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.tick_params(axis="y", labelsize=8)
fig.tight_layout(); print(A.save(fig, "02_chunks_per_document"))

## Overlap between consecutive chunks

Measured as: the first 60 characters of a chunk occur in the previous chunk of the same version (computed in SQL, `research/sql/chunk_metrics.sql`).

In [ ]:
nxt = active[active["chunk_index"] > 0]
rate = nxt["overlap_with_previous"].astype(float).mean()
share_top = per_doc["chunks"].nlargest(1).sum() / per_doc["chunks"].sum()
display(Markdown(f"""**Interpretation.** The configured overlap is {cfg['chunker']['overlap_tokens']} estimated tokens, but only
{rate:.1%} of non-first chunks ({int(nxt['overlap_with_previous'].sum())}/{len(nxt)}) begin with text found in the previous
chunk: the chunker breaks at headings/paragraphs and carries overlap only inside long sections, so most boundaries are
clean section breaks. Chunk sizes are skewed (median {active['chars'].median():.0f}, mean {active['chars'].mean():.0f}
characters; {int((active['chars'] < 250).sum())} chunks under 250 characters, minimum {active['chars'].min()}). One
document holds {share_top:.0%} of all active chunks, so corpus-level chunk statistics are dominated by it."""))